In [ ]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

### Область работы 1 (библиотеки)

In [52]:
# Данный блок в области 1 НЕ выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
#
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
#
# pip install


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, classification_report
)
 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.preprocessing import PolynomialFeatures
from scipy.stats import zscore

### Область работы 2 (поиск  модели .... )

In [2]:
data = pd.read_csv("train.csv")
df = data.copy()
X = df.drop(['quality','total sulfur dioxide','residual sugar']  ,axis=1 )
y = (df['quality'] >= 6).astype(int)

Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1
mask = ~((X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))).any(axis=1)

print("Выбросов удалено:", (~mask).sum())
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)

print("Новая форма X:", X.shape)
print("Новая форма y:", y.shape)

Выбросов удалено: 527
Новая форма X: (2261, 9)
Новая форма y: (2261,)


In [3]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.3, stratify=y_tmp, random_state=42)

In [8]:
base_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("sc", StandardScaler()), 
    ("clf", KNeighborsClassifier())
])

param_grid = [
    {
        "clf": [KNeighborsClassifier()],
        "clf__n_neighbors": [3,5,7,9 ,11,15],
        "clf__weights": ["uniform","distance"],
        "clf__p": [1,2],
        "poly__degree": [1,2,3,4]
    },
    {
        "clf": [LogisticRegression(max_iter=2000 )],
        "clf__C": [0.1, 1, 3, 10],
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs","liblinear"],
        "poly__degree": [1,2,3,4]
    },
    {
        "clf": [GaussianNB()],
        "clf__var_smoothing": [1e-9,1e-8,1e-7],
        "poly__degree": [1,2,3,4]
    },
    {
        "clf": [SVC(probability=True)],
        "clf__kernel": ["linear","rbf"],
        "clf__C": [0.5,1,3,10],
        "clf__gamma": ["scale","auto"],
        "poly__degree": [1,2,3,4]
    }
]

gs = GridSearchCV(
    estimator=base_pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=1
)

gs.fit(X_train, y_train)
print("Best params:", gs.best_params_)
print("Best CV F1 on train:", gs.best_score_)

best_pipe = gs.best_estimator_
proba_val = best_pipe.predict_proba(X_val)[:,1]
thresholds = np.linspace(0.01,1,100)
f1s = [f1_score(y_val,(proba_val>=t).astype(int)) for t in thresholds]
best_th = thresholds[np.argmax(f1s)]

print(f"Best thres hold on val: {best_th:.3f}, F1_val={np.max(f1s):.4f}")

proba_test = best_pipe.predict_proba(X_test)[:,1]
y_pred_test = (proba_test>=best_th).astype(int)

print("\nTEST METRICS ")
print("F1:", f1_score(y_test,y_pred_test))
print("Precision:", precision_score(y_test,y_pred_test))
print("Recall:", recall_score(y_test,y_pred_test))
print("ROC-AUC:", roc_auc_score(y_test,proba_test))
print("Confusion:\n", confusion_matrix(y_test,y_pred_test))
print("\nReport:\n", classification_report(y_test,y_pred_test))


best_pipe_final = clone(best_pipe)
best_pipe_final.fit(pd.concat([X_train,X_val]), pd.concat([y_train,y_val]))
proba_all = best_pipe_final.predict_proba(X)[:,1]
y_pred = (proba_all>=best_th).astype(int)

print("\nИтоговый y_pred shape:", y_pred.shape)


Fitting 5 folds for each of 204 candidates, totalling 1020 fits
Best params: {'clf': KNeighborsClassifier(), 'clf__n_neighbors': 15, 'clf__p': 1, 'clf__weights': 'distance', 'poly__degree': 4}
Best CV F1 on train: 0.8276448911820993
Best thres hold on val: 0.400, F1_val=0.8350

TEST METRICS 
F1: 0.8329355608591885
Precision: 0.7949886104783599
Recall: 0.87468671679198
ROC-AUC: 0.8904493376297888
Confusion:
 [[190  90]
 [ 50 349]]

Report:
               precision    recall  f1-score   support

           0       0.79      0.68      0.73       280
           1       0.79      0.87      0.83       399

    accuracy                           0.79       679
   macro avg       0.79      0.78      0.78       679
weighted avg       0.79      0.79      0.79       679


Итоговый y_pred shape: (2261,)
